# Load Library


In [15]:
import mlflow
import numpy as np
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
df=pd.read_csv("https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/refs/heads/main/tweet_emotions.csv")
df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [3]:
df.drop(columns="tweet_id",inplace=True)

In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def lemmatization(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

def remove_stop_words(text):
    return " ".join([word for word in str(text).split() if word not in stop_words])

def removing_numbers(text):
    return "".join([char for char in text if not char.isdigit()])

def lower_case(text):
    return " ".join([word.lower() for word in text.split()])

def remove_punctuation(text):
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    return text.strip()

def remove_urls(text):
    return re.sub(r"http\S+", "", text).strip()


def normalize_text(df: pd.DataFrame) -> pd.DataFrame:
    try:
        print("Starting text normalization")

        df['content'] = df['content'].astype(str)

        df['content'] = df['content'].apply(remove_urls)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_punctuation)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(lemmatization)

        print("Text normalization completed")
        return df

    except Exception as e:
        print(f"Error in text normalization: {e}")
        raise

In [5]:
df=normalize_text(df)

Starting text normalization
Text normalization completed


In [6]:
df=df[df['sentiment'].isin(['happiness','sadness'])]

In [7]:
df['sentiment'].replace({'sadness':0,'happiness':1},inplace=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_1780\2140401980.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'].replace({'sadness':0,'happiness':1},inplace=True)


In [8]:
X=df['content'] #input
y=df['sentiment'] #target

In [9]:
countvector=CountVectorizer(max_features=1000)
X=countvector.fit_transform(X)

In [10]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
import dagshub
import mlflow
mlflow.set_tracking_uri("https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/")
dagshub.init(repo_owner='rahulpatel16092005', repo_name='mlops-mini-project', mlflow=True)

Accessing as rahulpatel16092005

Initialized MLflow to track repo "rahulpatel16092005/mlops-mini-project"

Repository rahulpatel16092005/mlops-mini-project initialized!

In [12]:
mlflow.set_experiment("Logistic Regression Hyperparameter Tuning")

2026/03/29 14:54:17 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Hyperparameter Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/fd1cb1f33d274ceaa1be29799ffed1a0', creation_time=1774776259179, experiment_id='2', last_update_time=1774776259179, lifecycle_stage='active', name='Logistic Regression Hyperparameter Tuning', tags={}, workspace='default'>

In [13]:
params={
    'C':[0.01,0.1,1,10,100],
    'penalty':['l1', 'l2'],
    'solver':['liblinear']

}

In [16]:

grid_search = GridSearchCV(estimator=LogisticRegression(), param_grid=params, cv=5,scoring='accuracy',n_jobs=-1)
grid_search.fit(x_train, y_train)

c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displ

In [ ]:
grid_search.cv_results_['params']

{'mean_fit_time': array([0.05273395, 0.03887653, 0.02464156, 0.0325141 , 0.04172621,
        0.04285431, 0.02413368, 0.06915426, 0.02992005, 0.06490426]),
 'std_fit_time': array([0.02091599, 0.02357682, 0.01250578, 0.01640245, 0.02396415,
        0.01114787, 0.00434444, 0.01270751, 0.00879806, 0.00424677]),
 'mean_score_time': array([0.00862632, 0.00518332, 0.00382833, 0.00432472, 0.00525818,
        0.00282416, 0.00345893, 0.00292015, 0.00810781, 0.0024178 ]),
 'std_score_time': array([0.00517318, 0.00188817, 0.00156007, 0.00118363, 0.00208454,
        0.00083515, 0.00204146, 0.00105775, 0.01111946, 0.00086993]),
 'param_C': masked_array(data=[0.01, 0.01, 0.1, 0.1, 1.0, 1.0, 10.0, 10.0, 100.0,
                    100.0],
              mask=[False, False, False, False, False, False, False, False,
                    False, False],
        fill_value=1e+20),
 'param_penalty': masked_array(data=['l1', 'l2', 'l1', 'l2', 'l1', 'l2', 'l1', 'l2', 'l1',
                    'l2'],
            

In [19]:
with mlflow.start_run():
    mlflow.log_param("Vectorizer","Bag of words")
    mlflow.log_param("max_features",1000)
    mlflow.log_param("test_size",0.2)

    grid_search = GridSearchCV(estimator=LogisticRegression(), param_grid=params, cv=5,scoring='accuracy',n_jobs=-1)
    grid_search.fit(x_train, y_train)

    for param,mean_score,std_score in zip(grid_search.cv_results_['params'], grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['std_test_score']):
        with mlflow.start_run(run_name=f"LogisticRegression with {param}",nested=True):
                model=LogisticRegression(**param)
                model.fit(x_train,y_train)
                mlflow.log_param("model", "Logistic Regression")

                y_pred = model.predict(x_test)

                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred, average='weighted')
                recall = recall_score(y_test, y_pred, average='weighted')
                f1 = f1_score(y_test, y_pred, average='weighted')

                mlflow.log_param("C", param['C'])
                mlflow.log_param("penalty", param['penalty'])
                mlflow.log_param("solver", param['solver'])

                mlflow.log_metric("mean_cv_score", mean_score)
                mlflow.log_metric("std_cv_score", std_score)

                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", precision)
                mlflow.log_metric("recall", recall)
                mlflow.log_metric("f1_score", f1)
    
    best_params = grid_search.best_params_
    best_model = LogisticRegression(**best_params)
    best_model.fit(x_train, y_train)
    mlflow.log_param("params", best_params)
    mlflow.log_metric("best accuracy score", grid_search.best_score_)
    mlflow.sklearn.log_model(best_model, "Logistic Regression with Hyperparameter Tuning")

    notebook_path="exp3_bow_lor_hp.ipynb"
    import os
    os.system(f"jupyter nbconvert --to script {notebook_path}")
    mlflow.log_artifact(notebook_path)

c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_

🏃 View run LogisticRegression with {'C': 0.01, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2/runs/6a1ef8518f744262b6ee9f3ad0ef5119
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2


c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


🏃 View run LogisticRegression with {'C': 0.01, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2/runs/24a4c57e13714f5fa54c0da5e8fbf925
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2


c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


...

🏃 View run LogisticRegression with {'C': 100, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2/runs/ca060cb39b8b4185a97775e3cceebb94
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2


c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


🏃 View run LogisticRegression with {'C': 100, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2/runs/1158e5dea2844fbd9c20bb369a4f29a1
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2


c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
2026/03/29 15:11:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 15:11:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbit

🏃 View run beautiful-sponge-598 at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2/runs/0e270fbc12f04fd3a79d073ec3a29fb6
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/2
